# Bedrock Cost Controls — Guided Walkthrough

Walks through the full enforcement loop against a **deployed** `BedrockCostControlsStack`:

1. Preflight — confirm resources and invocation-logging state
2. Create an isolated demo IAM role (never touches a real user)
3. Show the pricing table and the cost math
4. Set a deliberately tiny budget so enforcement fires in seconds
5. Drive spend up through 50% and 80% alert thresholds
6. Breach the budget, watch the IAM deny policy get attached
7. **Prove** the deny works using the IAM policy simulator
8. Reset the budget and confirm the unlock
9. Clean up

## Prerequisites

- `cdk deploy` completed, and `python scripts/seed_pricing.py` run
- Credentials with: DynamoDB read/write, `lambda:InvokeFunction`, `iam:CreateRole`/`DeleteRole`/`GetRolePolicy`/`SimulatePrincipalPolicy`, `bedrock:GetModelInvocationLoggingConfiguration`
- To receive the SNS emails while running this, confirm the topic subscription **first**

## How the demo drives the system

We synthesize `ModelInvocationLog` records matching the documented Bedrock schema and hand them to the
enforcement Lambda in the same base64+gzip `awslogs` envelope CloudWatch Logs uses. Everything downstream
of the log-delivery hop is the real code path: real dedup, real pricing lookup, real atomic spend
accumulation, real IAM enforcement, real SNS.

Two delivery modes:

- `DELIVERY = "lambda"` (default) — invoke the Lambda directly. Fast and deterministic. **Recommended.**
- `DELIVERY = "logs"` — `PutLogEvents` into the real log group and let the subscription filter fire. Exercises
  the whole chain but adds the 5–30s log-delivery latency.

No real Bedrock spend is incurred either way.

---
## 0. Configuration

In [ ]:
import base64
import gzip
import json
import time
import uuid
from datetime import datetime, timezone
from decimal import Decimal

import boto3
from botocore.exceptions import ClientError

# ---- Environment ----
REGION = "us-east-1"

# ---- Deployed resource names (match the CDK stack defaults) ----
PRICING_TABLE = "BedrockModelPricing"
USAGE_TABLE = "BedrockUserUsage"
BUDGET_TABLE = "BedrockBudgetConfig"
DEDUP_TABLE = "BedrockInvocationDedup"
ENFORCEMENT_FN = "BedrockCostEnforcement"
RESET_FN = "BedrockBudgetReset"
LOG_GROUP = "/aws/bedrock/model-invocations"  # only used when DELIVERY == "logs"

# ---- Demo identity (a throwaway role, created below) ----
DEMO_ROLE = "BedrockCostControlsDemoRole"
DEMO_SESSION = "demo-user-01"   # becomes user_id via identity.arn parsing
DEMO_USER_ID = DEMO_SESSION

# ---- Demo budget: small enough to breach in six invocations ----
DEMO_DAILY_LIMIT = Decimal("0.50")
DEMO_MONTHLY_LIMIT = Decimal("100")
DEMO_THRESHOLDS = [Decimal("0.5"), Decimal("0.8")]

# ---- Model used for the simulated traffic ----
# A real model ID with authoritative pricing from the AmazonBedrockFoundationModels
# service code, including both prompt-cache TTL tiers. The `us.` prefix is a
# cross-region inference profile, which bills at the Global rate scope.
DEMO_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
TOKENS_IN = 5000
TOKENS_OUT = 5000

DELIVERY = "lambda"   # "lambda" or "logs"

# ---- Clients ----
session = boto3.Session(region_name=REGION)
ddb = session.resource("dynamodb")
lam = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
bedrock = session.client("bedrock")
sts = session.client("sts")

pricing_tbl = ddb.Table(PRICING_TABLE)
usage_tbl = ddb.Table(USAGE_TABLE)
budget_tbl = ddb.Table(BUDGET_TABLE)
dedup_tbl = ddb.Table(DEDUP_TABLE)

ACCOUNT_ID = sts.get_caller_identity()["Account"]
DEMO_ARN = f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/{DEMO_ROLE}/{DEMO_SESSION}"
RUN_ID = uuid.uuid4().hex[:8]   # keeps request IDs unique across reruns

print(f"Account : {ACCOUNT_ID}")
print(f"Region  : {REGION}")
print(f"Demo ARN: {DEMO_ARN}")
print(f"Run ID  : {RUN_ID}")

---
## 1. Preflight

Confirms the stack is deployed and pricing is seeded. Also reports the account's model invocation
logging configuration — worth showing on screen, because **this solution is entirely dependent on it.**

In [ ]:
def check(label, fn):
    try:
        detail = fn()
        print(f"  [ OK ] {label}{' — ' + detail if detail else ''}")
        return True
    except Exception as e:
        print(f"  [FAIL] {label} — {type(e).__name__}: {e}")
        return False

print("DynamoDB tables")
for name, tbl in [(PRICING_TABLE, pricing_tbl), (USAGE_TABLE, usage_tbl),
                  (BUDGET_TABLE, budget_tbl), (DEDUP_TABLE, dedup_tbl)]:
    check(name, lambda t=tbl: f"status={t.table_status}")

print("\nLambda functions")
for fn_name in [ENFORCEMENT_FN, RESET_FN]:
    check(fn_name, lambda f=fn_name: lam.get_function_configuration(FunctionName=f)["Runtime"])

print("\nSeed data")
check(f"pricing for {DEMO_MODEL_ID}",
      lambda: "present" if pricing_tbl.get_item(Key={"model_id": DEMO_MODEL_ID}).get("Item")
      else (_ for _ in ()).throw(RuntimeError("missing — run scripts/seed_pricing.py")))
check("DEFAULT budget config",
      lambda: "present" if budget_tbl.get_item(Key={"user_id": "DEFAULT"}).get("Item")
      else (_ for _ in ()).throw(RuntimeError("missing — run scripts/seed_pricing.py")))

print("\nBedrock model invocation logging (the dependency this whole solution rests on)")
try:
    cfg = bedrock.get_model_invocation_logging_configuration().get("loggingConfig", {})
    if not cfg:
        print("  [WARN] Not configured. Enforcement will never fire until this is enabled.")
    else:
        cw = cfg.get("cloudWatchConfig") or {}
        s3 = cfg.get("s3Config") or {}
        print(f"  CloudWatch log group : {cw.get('logGroupName', '(none)')}")
        print(f"  S3 bucket            : {s3.get('bucketName', '(none)')}")
        print(f"  text / image / embed / video : "
              f"{cfg.get('textDataDeliveryEnabled')} / {cfg.get('imageDataDeliveryEnabled')} / "
              f"{cfg.get('embeddingDataDeliveryEnabled')} / {cfg.get('videoDataDeliveryEnabled')}")
        if cw.get("logGroupName") and cw["logGroupName"] != LOG_GROUP:
            print(f"  [NOTE] Configured group differs from LOG_GROUP in this notebook ({LOG_GROUP}).")
except ClientError as e:
    print(f"  [WARN] Could not read logging config: {e}")

print("\nNote: invocation logging covers the bedrock-runtime endpoint only.")
print("Traffic on bedrock-mantle (OpenAI Chat Completions / Responses, Anthropic Messages)")
print("is NOT captured, and therefore is NOT metered or enforced by this solution.")

---
## 2. Create the demo IAM role

Enforcement works by attaching an inline **deny** policy to the caller's IAM role. To keep the demo safe
we create a dedicated throwaway role with a trust policy nobody can actually assume. The role only needs
to *exist* for `PutRolePolicy` to succeed and for the policy simulator to evaluate against it.

In [ ]:
TRUST = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Deny",
        "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
        "Action": "sts:AssumeRole",
    }],
}

ALLOW_BEDROCK = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream",
                   "bedrock:Converse", "bedrock:ConverseStream"],
        "Resource": "*",
    }],
}

try:
    iam.create_role(
        RoleName=DEMO_ROLE,
        AssumeRolePolicyDocument=json.dumps(TRUST),
        Description="Throwaway role for Bedrock cost-controls demo. Safe to delete.",
        Tags=[{"Key": "purpose", "Value": "bedrock-cost-controls-demo"}],
    )
    print(f"Created role {DEMO_ROLE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {DEMO_ROLE} already exists — reusing")
    else:
        raise

# Baseline allow, so the policy simulator has something for the deny to override.
iam.put_role_policy(
    RoleName=DEMO_ROLE,
    PolicyName="DemoAllowBedrock",
    PolicyDocument=json.dumps(ALLOW_BEDROCK),
)
print("Attached baseline DemoAllowBedrock policy")

time.sleep(8)  # let IAM settle before the simulator reads it
print("Ready.")

---
## 3. Pricing table and cost math

Pricing comes from the `BedrockModelPricing` table, refreshed daily by the Pricing Sync Lambda from the
AWS Price List API. Here's what a single simulated invocation will cost.

In [ ]:
item = pricing_tbl.get_item(Key={"model_id": DEMO_MODEL_ID}).get("Item")
if not item:
    raise SystemExit(f"No pricing for {DEMO_MODEL_ID}. Run scripts/seed_pricing.py first.")

p_in = Decimal(str(item["input_price_per_1k_tokens"]))
p_out = Decimal(str(item["output_price_per_1k_tokens"]))

# Prompt-cache rates. Read these from the table rather than deriving them from
# multipliers, so later sections verify against published rates instead of
# reproducing the same assumption the Lambda makes.
CACHE_READ_MULTIPLIER = Decimal("0.10")     # fallback only
CACHE_WRITE_MULTIPLIER = Decimal("1.25")    # fallback only
CACHE_WRITE_1H_MULTIPLIER = Decimal("2.00")  # fallback only


def _rate(column, multiplier):
    """Published rate if present, else the documented multiplier of input."""
    published = item.get(column)
    if published is not None:
        return Decimal(str(published)), "published"
    return p_in * multiplier, "derived"


p_cache_read, src_read = _rate("cache_read_price_per_1k_tokens", CACHE_READ_MULTIPLIER)
p_cache_w5m, src_w5m = _rate("cache_write_price_per_1k_tokens", CACHE_WRITE_MULTIPLIER)
p_cache_w1h, src_w1h = _rate("cache_write_1h_price_per_1k_tokens", CACHE_WRITE_1H_MULTIPLIER)

cost_in = (Decimal(TOKENS_IN) / 1000) * p_in
cost_out = (Decimal(TOKENS_OUT) / 1000) * p_out
COST_PER_INVOCATION = cost_in + cost_out

print(f"Model                : {DEMO_MODEL_ID}")
print(f"Pricing source       : {item.get('pricing_service_code', item.get('source', 'unknown'))}")
print(f"Rate scope           : {item.get('rate_scope', '(n/a)')}")
print(f"Cache pricing source : {item.get('cache_pricing_source', '(none)')}")
print(f"Effective date       : {item.get('effective_date')}")
print()
print(f"{'component':<26} {'per 1K':>12} {'ratio':>8}  source")
print("-" * 62)
print(f"{'input':<26} ${p_in:>11} {'1.00x':>8}  published")
print(f"{'output':<26} ${p_out:>11} {'':>8}  published")
print(f"{'cache read':<26} ${p_cache_read:>11} "
      f"{str(p_cache_read / p_in) + 'x':>8}  {src_read}")
print(f"{'cache write (5m TTL)':<26} ${p_cache_w5m:>11} "
      f"{str(p_cache_w5m / p_in) + 'x':>8}  {src_w5m}")
print(f"{'cache write (1h TTL)':<26} ${p_cache_w1h:>11} "
      f"{str(p_cache_w1h / p_in) + 'x':>8}  {src_w1h}")
print()
print(f"Simulated invocation : {TOKENS_IN:,} in + {TOKENS_OUT:,} out")
print(f"  input  cost        : ${cost_in}")
print(f"  output cost        : ${cost_out}")
print(f"  total              : ${COST_PER_INVOCATION}")
print()
n_breach = int(DEMO_DAILY_LIMIT / COST_PER_INVOCATION) + 1
print(f"Daily limit ${DEMO_DAILY_LIMIT} will be breached on invocation #{n_breach}")
print(f"  50% alert at ${DEMO_DAILY_LIMIT * Decimal('0.5')}")
print(f"  80% alert at ${DEMO_DAILY_LIMIT * Decimal('0.8')}")

---
## 4. Set the demo budget and clear prior state

Per-user budgets live in `BedrockBudgetConfig`. Users with no row fall back to the `DEFAULT` row.

In [ ]:
budget_tbl.put_item(Item={
    "user_id": DEMO_USER_ID,
    "daily_limit_usd": DEMO_DAILY_LIMIT,
    "monthly_limit_usd": DEMO_MONTHLY_LIMIT,
    "alert_thresholds": DEMO_THRESHOLDS,
    "team": "demo",
})
print(f"Budget set for {DEMO_USER_ID}: daily=${DEMO_DAILY_LIMIT}, monthly=${DEMO_MONTHLY_LIMIT}, "
      f"thresholds={[str(t) for t in DEMO_THRESHOLDS]}")

# Clear any usage left over from a previous run
usage_tbl.put_item(Item={
    "user_id": DEMO_USER_ID,
    "daily_spend_usd": Decimal("0"),
    "monthly_spend_usd": Decimal("0"),
    "daily_invocation_count": 0,
    "is_denied": False,
    "iam_role_name": DEMO_ROLE,
    "iam_user_name": "",
    "principal_type": "role",
})
print("Usage accumulators zeroed")

# Drop a stale deny policy if a previous run left one behind
try:
    iam.delete_role_policy(RoleName=DEMO_ROLE, PolicyName=f"BudgetExceeded-{DEMO_USER_ID}")
    print("Removed stale deny policy from previous run")
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchEntity":
        raise
    print("No stale deny policy present")

---
## 5. Synthetic invocation log records

These match the documented `ModelInvocationLog` schema. Note that `input.inputTokenCount` and
`output.outputTokenCount` are populated by **Bedrock**, not the model provider — which is why the same
parsing works across Anthropic, Nova, Llama, Mistral, DeepSeek and the rest.

In [ ]:
def make_record(seq, model_id=DEMO_MODEL_ID, tokens_in=TOKENS_IN, tokens_out=TOKENS_OUT,
                arn=DEMO_ARN, operation="Converse", cache_read=0, cache_write=0,
                include_usage=True):
    """
    Build one ModelInvocationLog record.

    By default the response body carries a `usage` object in Converse shape, which
    is what real Bedrock records contain. The enforcement Lambda reads cache token
    counts from there, so including it exercises the primary code path.

    Set include_usage=False to omit it and deliberately exercise the envelope-only
    fallback, where cache activity is invisible. That path is expected to raise the
    EnvelopeOnlyMetering metric.

    Note the Converse convention: `usage.inputTokens` EXCLUDES cache tokens, so
    tokens_in is the non-cached remainder and cache_read / cache_write are additive.
    """
    output_body = {"note": "example response"}
    if include_usage:
        output_body["usage"] = {
            "inputTokens": tokens_in,
            "outputTokens": tokens_out,
            "totalTokens": tokens_in + tokens_out + cache_read + cache_write,
            "cacheReadInputTokens": cache_read,
            "cacheWriteInputTokens": cache_write,
        }

    return {
        "schemaType": "ModelInvocationLog",
        "schemaVersion": "1.0",
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "accountId": ACCOUNT_ID,
        "region": REGION,
        "requestId": f"demo-{RUN_ID}-{seq:04d}",
        "operation": operation,
        "modelId": model_id,
        "identity": {"arn": arn},
        "input": {"inputContentType": "application/json",
                  "inputBodyJson": {"note": "example payload"},
                  # Envelope count. Bedrock excludes cache tokens from this field.
                  "inputTokenCount": tokens_in},
        "output": {"outputContentType": "application/json",
                   "outputBodyJson": output_body,
                   "outputTokenCount": tokens_out},
    }


def encode_awslogs(records):
    """Wrap records in the base64+gzip envelope CloudWatch Logs delivers to Lambda."""
    now_ms = int(time.time() * 1000)
    payload = {
        "messageType": "DATA_MESSAGE",
        "owner": ACCOUNT_ID,
        "logGroup": LOG_GROUP,
        "logStream": "aws/bedrock/modelinvocations",
        "subscriptionFilters": ["BedrockInvocationFilter"],
        "logEvents": [
            {"id": str(uuid.uuid4()), "timestamp": now_ms, "message": json.dumps(r)}
            for r in records
        ],
    }
    return {"awslogs": {"data": base64.b64encode(
        gzip.compress(json.dumps(payload).encode())).decode()}}


def send(records):
    """Deliver records to the enforcement Lambda."""
    if DELIVERY == "lambda":
        resp = lam.invoke(
            FunctionName=ENFORCEMENT_FN,
            InvocationType="RequestResponse",
            Payload=json.dumps(encode_awslogs(records)).encode(),
        )
        body = json.loads(resp["Payload"].read() or b"{}")
        if resp.get("FunctionError"):
            raise RuntimeError(f"Lambda error: {body}")
        return body
    elif DELIVERY == "logs":
        stream = f"demo-{RUN_ID}"
        try:
            logs.create_log_stream(logGroupName=LOG_GROUP, logStreamName=stream)
        except ClientError as e:
            if e.response["Error"]["Code"] != "ResourceAlreadyExistsException":
                raise
        logs.put_log_events(
            logGroupName=LOG_GROUP, logStreamName=stream,
            logEvents=[{"timestamp": int(time.time() * 1000), "message": json.dumps(r)}
                       for r in records],
        )
        return {"delivered_via": "PutLogEvents", "count": len(records)}
    raise ValueError(f"Unknown DELIVERY mode: {DELIVERY}")


def read_usage():
    return usage_tbl.get_item(Key={"user_id": DEMO_USER_ID}).get("Item", {})


print(f"Delivery mode: {DELIVERY}")
print("\nSample record:")
print(json.dumps(make_record(0), indent=2)[:900] + "\n...")

---
## 6. Drive spend up through the alert thresholds

Each row is a real pass through the enforcement Lambda. Watch `pct` climb; alerts fire on the
invocation that *crosses* a threshold boundary, so each one fires exactly once.

In [ ]:
print(f"{'#':>3}  {'cost':>10}  {'daily spend':>12}  {'pct':>6}  {'count':>6}  {'denied':>7}  note")
print("-" * 86)

prev_pct = Decimal("0")
for i in range(1, n_breach + 1):
    send([make_record(i)])
    if DELIVERY == "logs":
        time.sleep(20)

    u = read_usage()
    spend = Decimal(str(u.get("daily_spend_usd", 0)))
    pct = (spend / DEMO_DAILY_LIMIT) * 100
    denied = bool(u.get("is_denied", False))

    note = ""
    for t in DEMO_THRESHOLDS:
        boundary = t * 100
        if prev_pct < boundary <= pct:
            note = f"<- {int(boundary)}% ALERT sent to SNS"
    if denied:
        note = "<- BUDGET EXCEEDED, IAM deny attached"

    print(f"{i:>3}  ${COST_PER_INVOCATION:>9}  ${spend:>11}  {pct:>5.1f}%  "
          f"{int(u.get('daily_invocation_count', 0)):>6}  {str(denied):>7}  {note}")
    prev_pct = pct

    if denied:
        break

print("-" * 86)
print(f"Final daily spend: ${Decimal(str(read_usage().get('daily_spend_usd', 0)))} "
      f"against a ${DEMO_DAILY_LIMIT} limit")

---
## 7. Show the enforcement artifact

The deny is a plain IAM inline policy named `BudgetExceeded-{user_id}`. Predictable name, easy to audit,
easy to remove.

In [ ]:
policy_name = f"BudgetExceeded-{DEMO_USER_ID}"
policy_doc = None
try:
    resp = iam.get_role_policy(RoleName=DEMO_ROLE, PolicyName=policy_name)
    policy_doc = resp["PolicyDocument"]
    print(f"Policy '{policy_name}' on role '{DEMO_ROLE}':\n")
    print(json.dumps(policy_doc, indent=2))
except ClientError as e:
    print(f"Deny policy not found: {e}")

# The deny MUST be scoped to this one role session. Without the aws:userid
# condition it would apply to every principal assuming this role, so on a shared
# IAM Identity Center role one user over budget would block the whole team.
if policy_doc:
    stmt = policy_doc["Statement"][0]
    cond = stmt.get("Condition", {}).get("StringLike", {}).get("aws:userid")
    print("\nScoping check")
    print(f"  Effect                : {stmt['Effect']}")
    print(f"  Resource              : {stmt['Resource']}")
    print(f"  aws:userid condition  : {cond or '*** ABSENT ***'}")
    if cond and cond.endswith(f":{DEMO_USER_ID}"):
        print(f"  -> SCOPED to session '{DEMO_USER_ID}' only. Other sessions on "
              f"'{DEMO_ROLE}' are unaffected.")
    else:
        print("  -> UNSCOPED. This would deny Bedrock to every principal assuming "
              "this role.")
        print("     Check that the Lambda has iam:GetRole and REQUIRE_SCOPED_DENY=true.")

print("\nInline policies now on the role:")
for p in iam.list_role_policies(RoleName=DEMO_ROLE)["PolicyNames"]:
    print(f"  - {p}")

u = read_usage()
print(f"\nUsage record:")
print(f"  is_denied         : {u.get('is_denied')}")
print(f"  denied_at         : {u.get('denied_at')}")
print(f"  role              : {u.get('iam_role_name')}")
print(f"  last invoke       : {u.get('last_invocation_ts')}")
print(f"  last_usage_source : {u.get('last_usage_source', '(not set)')}")
print(f"  alerts_sent       : {sorted(u.get('alerts_sent', [])) or '(none)'}")
print(f"\nToken counters accumulated:")
for f in ("daily_input_tokens", "daily_output_tokens",
          "daily_cache_read_tokens", "daily_cache_write_tokens"):
    print(f"  {f:<26} {int(u.get(f, 0)):>10,}")

---
## 8. Prove the deny blocks the right principal — and only that principal

Rather than asserting it, evaluate it. `iam:SimulatePrincipalPolicy` runs the real IAM policy evaluation
engine against the role's current policies.

Two things need proving, and the second matters more:

1. The over-budget session is denied every Bedrock invocation action.
2. **Every other session on the same role still works.**

Point 2 is the whole reason the deny carries a condition on `aws:userid`. On a shared IAM Identity Center
role an unscoped deny would have blocked the entire team.

### Why the simulator needs a session context

`aws:userid` for a role session is `<role-unique-id>:<session-name>`. The simulator has no session of its
own, so that key has to be passed in via `ContextEntries`. If you omit it, the condition cannot be
evaluated, the `Deny` is skipped, and the simulator returns **`allowed`** — which looks like a broken
control but is actually the scoping behaving correctly. The cell below shows both, so the distinction is
visible rather than confusing.

In [ ]:
ACTIONS = ["bedrock:InvokeModel", "bedrock:Converse", "bedrock:ConverseStream"]
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{DEMO_ROLE}"

# The deny is scoped with a condition on aws:userid, whose value for a role session
# is "<role-unique-id>:<session-name>". The simulator has no session of its own, so
# that key must be supplied explicitly. Omit it and the condition cannot be
# evaluated, the Deny does not apply, and the result is correctly "allowed" —
# which looks like a broken control but is actually the scoping working.
role_id = iam.get_role(RoleName=DEMO_ROLE)["Role"]["RoleId"]
DENIED_USERID = f"{role_id}:{DEMO_USER_ID}"
OTHER_USERID = f"{role_id}:another-teammate"


def simulate(aws_userid=None):
    """Evaluate ACTIONS against the role, optionally as a specific session."""
    kwargs = {"PolicySourceArn": role_arn, "ActionNames": ACTIONS}
    if aws_userid:
        kwargs["ContextEntries"] = [{"ContextKeyName": "aws:userid",
                                    "ContextKeyValues": [aws_userid],
                                    "ContextKeyType": "string"}]
    res = iam.simulate_principal_policy(**kwargs)
    return {
        r["EvalActionName"]: (
            r["EvalDecision"],
            ", ".join(s.get("SourcePolicyId", "?")
                      for s in r.get("MatchedStatements", [])) or "-",
        )
        for r in res["EvaluationResults"]
    }


time.sleep(10)  # IAM is eventually consistent

print(f"Role      : {role_arn}")
print(f"Role ID   : {role_id}")
print(f"aws:userid of the over-budget session: {DENIED_USERID}\n")

SCENARIOS = [
    (f"the over-budget session ({DEMO_USER_ID})", DENIED_USERID, "explicitDeny"),
    ("a different session on the SAME role", OTHER_USERID, "allowed"),
]

print(f"{'scenario':<38} {'action':<28} {'decision':<14} matched policy")
print("-" * 108)

all_ok = True
for label, userid, expected in SCENARIOS:
    outcome = simulate(userid)
    for n, (action, (decision, matched)) in enumerate(sorted(outcome.items())):
        print(f"{label if n == 0 else '':<38} {action:<28} {decision:<14} {matched}")
    ok = {d for d, _ in outcome.values()} == {expected}
    all_ok = all_ok and ok
    print(f"{'':<38} -> expected {expected}: {'PASS' if ok else 'FAIL'}\n")

print("-" * 108)
if all_ok:
    print("CONFIRMED by AWS's own policy evaluator:")
    print(f"  - {DEMO_USER_ID} is explicitly denied every Bedrock invocation action.")
    print("  - Every OTHER session on the same role is unaffected.")
    print("\nThe second point is what makes this safe on a shared IAM Identity Center")
    print("role. An unscoped deny would have blocked the entire team.")
else:
    print("Unexpected result. Check that the deny policy is attached and that the")
    print("Lambda had iam:GetRole so it could build the scoped condition.")

# For contrast: the same simulation with no session context at all.
no_ctx = {d for d, _ in simulate().values()}
print(f"\nFor contrast, simulating with NO session context returns: {no_ctx}")
print("That is expected. Without aws:userid the scoped condition cannot match, so")
print("the Deny is skipped. It is not evidence that enforcement failed.")

---
## 9. Reset and unlock

The Budget Reset Lambda runs on an EventBridge schedule (daily and monthly). Invoking it manually here
shows the recovery path: accumulators cleared, deny policy removed, summary published to SNS.

In [ ]:
resp = lam.invoke(
    FunctionName=RESET_FN,
    InvocationType="RequestResponse",
    Payload=json.dumps({"reset_type": "daily"}).encode(),
)
print("Reset Lambda result:")
print(json.dumps(json.loads(resp["Payload"].read() or b"{}"), indent=2))

u = read_usage()
print(f"\nUsage after reset:")
print(f"  daily_spend_usd        : {u.get('daily_spend_usd')}")
print(f"  daily_invocation_count : {u.get('daily_invocation_count')}")
print(f"  monthly_spend_usd      : {u.get('monthly_spend_usd')}   (daily reset leaves this intact)")
print(f"  is_denied              : {u.get('is_denied')}")

# Everything the enforcement Lambda accumulates must be cleared, not just spend.
# If alerts_sent survived a reset, each user would receive a given threshold alert
# once and then never again — a silent, permanent regression.
checks = {
    "daily_spend_usd == 0": Decimal(str(u.get("daily_spend_usd", -1))) == 0,
    "daily_invocation_count == 0": int(u.get("daily_invocation_count", -1)) == 0,
    "is_denied is False": u.get("is_denied") is False,
    "alerts_sent cleared": not u.get("alerts_sent"),
    "daily_input_tokens == 0": int(u.get("daily_input_tokens", -1)) == 0,
    "daily_output_tokens == 0": int(u.get("daily_output_tokens", -1)) == 0,
    "daily_cache_read_tokens == 0": int(u.get("daily_cache_read_tokens", -1)) == 0,
    "daily_cache_write_tokens == 0": int(u.get("daily_cache_write_tokens", -1)) == 0,
}
print("\nReset completeness")
for label, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {label}")
if not all(checks.values()):
    print("\n  Some state survived the reset. Threshold alerts and token accounting")
    print("  will be wrong for the next period. Check budget_reset/handler.py.")

print("\nInline policies on the role:")
for p in iam.list_role_policies(RoleName=DEMO_ROLE)["PolicyNames"]:
    print(f"  - {p}")

time.sleep(10)

# Simulate as the previously-denied session. Passing the aws:userid context is
# essential: without it the result would be "allowed" whether or not the deny was
# actually removed, so the check would prove nothing.
after = simulate(DENIED_USERID)
print(f"\nRe-simulated as {DEMO_USER_ID} after unlock:")
for action, (decision, matched) in sorted(after.items()):
    print(f"  {action:<28} {decision:<14} {matched}")

unlocked = {d for d, _ in after.values()} == {"allowed"}
print()
print("PASS: the previously-denied session can invoke Bedrock again." if unlocked
      else "FAIL: still denied. Check that the reset removed the inline policy.")

---
## 10. Optional — multi-provider parsing

One record per provider, to show that a single parser handles all of them because Bedrock populates the
token counts. Also shows the two places the token model breaks down: an embeddings model (no output
tokens) and an image model (no tokens at all, so it prices at **$0.00** — a real gap, not a rounding issue).

In [ ]:
PROBES = [
    ("anthropic.claude-sonnet-4-v1:0",        2000, 1000, "Anthropic"),
    ("amazon.nova-pro-v1:0",                  2000, 1000, "Amazon Nova"),
    ("meta.llama3-3-70b-instruct-v1:0",       2000, 1000, "Meta Llama"),
    ("mistral.mistral-large-2407-v1:0",       2000, 1000, "Mistral"),
    ("deepseek.r1-v1:0",                      2000, 1000, "DeepSeek"),
    ("amazon.titan-embed-text-v2:0",          2000,    0, "Embeddings - no output tokens"),
    ("amazon.nova-canvas-v1:0",                  0,    0, "Image gen - NOT token priced"),
]

PROBE_USER = "demo-provider-probe"
probe_arn = f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/{DEMO_ROLE}/{PROBE_USER}"

budget_tbl.put_item(Item={
    "user_id": PROBE_USER,
    "daily_limit_usd": Decimal("1000"),      # high, so nothing gets denied here
    "monthly_limit_usd": Decimal("10000"),
    "alert_thresholds": [Decimal("0.99")],
})
usage_tbl.put_item(Item={
    "user_id": PROBE_USER, "daily_spend_usd": Decimal("0"),
    "monthly_spend_usd": Decimal("0"), "daily_invocation_count": 0,
    "is_denied": False, "iam_role_name": DEMO_ROLE,
    "iam_user_name": "", "principal_type": "role",
})

print(f"{'model_id':<42} {'in':>6} {'out':>6} {'priced':>9}  {'delta $':>12}  note")
print("-" * 104)

running = Decimal("0")
for seq, (mid, ti, to, label) in enumerate(PROBES, start=9000):
    priced = "yes" if pricing_tbl.get_item(Key={"model_id": mid}).get("Item") else "FALLBACK"
    send([make_record(seq, model_id=mid, tokens_in=ti, tokens_out=to, arn=probe_arn)])
    if DELIVERY == "logs":
        time.sleep(20)
    total = Decimal(str(usage_tbl.get_item(Key={"user_id": PROBE_USER})
                        .get("Item", {}).get("daily_spend_usd", 0)))
    delta = total - running
    running = total
    flag = "  <-- $0.00, silently untracked" if delta == 0 else ""
    print(f"{mid:<42} {ti:>6} {to:>6} {priced:>9}  ${delta:>11}  {label}{flag}")

print("-" * 104)
print(f"Total attributed to {PROBE_USER}: ${running}")
print("\nTakeaway: the envelope is provider-agnostic because Bedrock writes the token counts.")
print("But anything not priced per token (image, video, and imported/provisioned models)")
print("lands at $0.00 and escapes enforcement entirely.")

---
## 11. Prompt caching — verifying cache tokens are metered

**Status: implemented.** This was the largest accuracy gap in the original implementation, and it hit
agentic coding tools (Claude Code, Kiro, Cline) hardest because cached context dominates their token
volume. The enforcement Lambda now meters cache reads and writes. This section explains the problem and
then verifies the fix against the deployed stack.

### Why it happened

The enforcement Lambda reads `input.inputTokenCount` from the log envelope. Per the Bedrock docs:

> When prompt caching is enabled, the `inputTokens` field represents only the non-cached input tokens
> (tokens that were not read from or written to the cache). To calculate the total input tokens sent in a
> request, use the following formula:
> `total input tokens = inputTokens + cacheReadInputTokens + cacheWriteInputTokens`

The CloudWatch reference agrees: `InputTokenCount` is the number of input tokens processed by the model
**excluding cached tokens**.

So the field we meter on is the *residual* after cache activity is removed. For a Claude Code turn that
residual is roughly just the new user message, while the large cached prefix — the part that drives the
bill — sits in fields the handler never reads.

### Where the missing data actually lives

It is in the log, one level deeper. The entry carries `output.outputBodyJson` (the response body, up to
100 KB), and that body contains the `usage` object:

| API shape | Cache fields | `inputTokens` semantics |
|---|---|---|
| Converse / ConverseStream | `cacheReadInputTokens`, `cacheWriteInputTokens`, `cacheDetails` | **Excludes** cache |
| Anthropic native (InvokeModel) — *Claude Code's path* | `cache_read_input_tokens`, `cache_creation_input_tokens` | **Excludes** cache |
| OpenAI-compatible | `input_tokens_details.cached_tokens`, `.cache_write_tokens` | **Includes** cache |

Note the last row inverts. Applying one global formula across providers double-counts the cached prefix
on OpenAI models. Key on the response shape, not a single formula.

### The cell below

Sends three realistic Claude Code shaped records through the **real, deployed** Lambda and checks that
the cost it attributes now includes cache activity. The `envelope-only` column shows what the original
implementation would have charged, so the size of the correction is visible.

Requires the current code to be deployed (`cdk deploy`). Against an older deployment the `charged`
column will match `envelope-only` and the checks will fail — which is itself a useful deployment test.

In [ ]:
# Cache rates come from section 3, which reads them from the pricing table and only
# falls back to multipliers when a model has no published rate. That makes the
# comparison below a genuine check against published pricing rather than a restatement
# of the same assumption the Lambda makes.

CC_USER = "demo-claude-code"
cc_arn = f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/{DEMO_ROLE}/{CC_USER}"

# label, residual (non-cached) input, cache read, cache write, output
SCENARIOS = [
    ("conventional app call, no caching", 2000, 0, 0, 900),
    ("Claude Code, warm cache", 350, 120_000, 0, 900),
    ("Claude Code, cold cache", 350, 0, 120_000, 900),
]


def make_cache_record(seq, residual_in, cache_read, cache_write, out):
    """A record shaped like Claude Code's InvokeModel call against Anthropic native."""
    r = make_record(seq, tokens_in=residual_in, tokens_out=out, arn=cc_arn, operation="InvokeModel")
    r["output"]["outputBodyJson"] = {
        "usage": {
            "input_tokens": residual_in,
            "output_tokens": out,
            "cache_read_input_tokens": cache_read,
            "cache_creation_input_tokens": cache_write,
        }
    }
    return r


def extract_token_usage(record):
    """PROPOSED FIX. Normalizes usage across Converse, Anthropic-native and OpenAI shapes.

    Returns cache tokens separated from the non-cached residual. Anthropic/Converse report
    inputTokens EXCLUDING cache tokens; OpenAI models report input_tokens INCLUDING them.
    """
    usage = (record.get("output", {}).get("outputBodyJson") or {}).get("usage") or {}

    if "cacheReadInputTokens" in usage or "cacheWriteInputTokens" in usage:
        return {"input": usage.get("inputTokens", 0), "output": usage.get("outputTokens", 0),
                "cache_read": usage.get("cacheReadInputTokens", 0),
                "cache_write": usage.get("cacheWriteInputTokens", 0), "source": "converse"}

    if "cache_read_input_tokens" in usage or "cache_creation_input_tokens" in usage:
        return {"input": usage.get("input_tokens", 0), "output": usage.get("output_tokens", 0),
                "cache_read": usage.get("cache_read_input_tokens", 0),
                "cache_write": usage.get("cache_creation_input_tokens", 0),
                "source": "anthropic_native"}

    details = usage.get("input_tokens_details") or {}
    if details:
        cached = details.get("cached_tokens", 0)
        written = details.get("cache_write_tokens", 0)
        return {"input": max(usage.get("input_tokens", 0) - cached - written, 0),
                "output": usage.get("output_tokens", 0),
                "cache_read": cached, "cache_write": written, "source": "openai"}

    # Envelope only — cache activity invisible. Emit a metric on this in production.
    return {"input": record.get("input", {}).get("inputTokenCount", 0),
            "output": record.get("output", {}).get("outputTokenCount", 0),
            "cache_read": 0, "cache_write": 0, "source": "envelope_only"}


def corrected_cost(u, w1h=0):
    """
    Cache-aware cost, using the rates read from the pricing table in section 3.

    `w1h` is the portion of cache writes on the 1-hour TTL tier; the remainder is
    priced at the 5-minute rate. Records built by make_cache_record carry no TTL
    breakdown, so they default to 5-minute per the documented behaviour.
    """
    w5m = max(u["cache_write"] - w1h, 0)
    return (
        (Decimal(u["input"]) / 1000) * p_in
        + (Decimal(u["output"]) / 1000) * p_out
        + (Decimal(u["cache_read"]) / 1000) * p_cache_read
        + (Decimal(w5m) / 1000) * p_cache_w5m
        + (Decimal(w1h) / 1000) * p_cache_w1h
    )


# Headroom so nothing gets denied during this measurement
budget_tbl.put_item(Item={"user_id": CC_USER, "daily_limit_usd": Decimal("1000"),
                          "monthly_limit_usd": Decimal("10000"),
                          "alert_thresholds": [Decimal("0.99")]})

def envelope_only_cost(rec):
    """What the original two-term implementation would have charged."""
    ti = rec.get("input", {}).get("inputTokenCount", 0)
    to = rec.get("output", {}).get("outputTokenCount", 0)
    return (Decimal(ti) / 1000) * p_in + (Decimal(to) / 1000) * p_out


print(f"Model {DEMO_MODEL_ID}")
print(f"  input ${p_in}/1K  output ${p_out}/1K")
print(f"  cache read ${p_cache_read}/1K ({src_read})   "
      f"cache write 5m ${p_cache_w5m}/1K ({src_w5m})\n")
print(f"{'scenario':<34} {'envelope-only':>14} {'charged':>13} {'expected':>13} {'recovered':>11}  ok")
print("-" * 94)

totals = {"envelope": Decimal("0"), "charged": Decimal("0"), "expected": Decimal("0")}
all_ok = True

for n, (label, r_in, c_read, c_write, out) in enumerate(SCENARIOS, start=9500):
    # zero the accumulator so each scenario is measured in isolation
    usage_tbl.put_item(Item={"user_id": CC_USER, "daily_spend_usd": Decimal("0"),
                             "monthly_spend_usd": Decimal("0"), "daily_invocation_count": 0,
                             "is_denied": False, "iam_role_name": DEMO_ROLE,
                             "iam_user_name": "", "principal_type": "role"})

    rec = make_cache_record(n, r_in, c_read, c_write, out)
    send([rec])
    if DELIVERY == "logs":
        time.sleep(20)

    item = usage_tbl.get_item(Key={"user_id": CC_USER}).get("Item", {})
    charged = Decimal(str(item.get("daily_spend_usd", 0)))
    expected = corrected_cost(extract_token_usage(rec))
    envelope = envelope_only_cost(rec)

    # Allow a small tolerance: the deployed Lambda may be using real Price List
    # cache rates while this cell derives them from the documented multipliers.
    ok = expected == 0 or abs(charged - expected) / expected < Decimal("0.02")
    all_ok = all_ok and ok

    for k, v in (("envelope", envelope), ("charged", charged), ("expected", expected)):
        totals[k] += v

    print(f"{label:<34} ${envelope:>13.6f} ${charged:>12.6f} ${expected:>12.6f} "
          f"${charged - envelope:>10.6f}  {'PASS' if ok else 'FAIL'}")

print("-" * 94)
recovered = totals["charged"] - totals["envelope"]
factor = (totals["charged"] / totals["envelope"]) if totals["envelope"] else Decimal("0")
print(f"{'TOTAL':<34} ${totals['envelope']:>13.6f} ${totals['charged']:>12.6f} "
      f"${totals['expected']:>12.6f} ${recovered:>10.6f}")
print()
print(f"Cache-driven cost recovered across three requests: ${recovered:.6f} ({factor:.1f}x)")
print()

if all_ok:
    print("VERIFIED: the deployed Lambda meters prompt cache reads and writes.")
else:
    print("MISMATCH: charged does not include cache activity.")
    print("  The deployed Lambda is likely running the older two-term cost model.")
    print("  Redeploy with 'cdk deploy' and rerun this cell.")

print()
print("Row 1 is the regression guard: uncached traffic must price identically to before.")
print("Cold-cache writes are the worst case — they bill at a premium and were fully invisible.")
print("Claude Code rewrites cache often (context growth, 5-min TTL expiry), so cold writes recur.")

# Per-component token counters now recorded on the usage item
print("\nToken counters on the usage record (supports CloudWatch reconciliation):")
for field in ("daily_input_tokens", "daily_output_tokens",
              "daily_cache_read_tokens", "daily_cache_write_tokens"):
    print(f"  {field:<26} {int(item.get(field, 0)):>10,}")
print(f"  {'last_usage_source':<26} {item.get('last_usage_source', '(not set)'):>10}")

### Does cache spend alone trigger enforcement?

Metering cache tokens only matters if it actually feeds enforcement. This is the scenario that motivated
the fix: a coding assistant whose cost is almost entirely cache writes, with a trivial amount of new
input. Under envelope-only metering such a request costs a fraction of a cent and never trips a budget,
no matter how many times it repeats.

The cell below sets a tight budget and sends a single cache-heavy request. It should breach immediately.

It then walks all four response shapes through the deployed Lambda and checks that each is classified
correctly, since choosing the wrong branch would either miss cache tokens or double-count them.

Finally it verifies that the two cache-write TTL tiers are priced separately. Anthropic bills a
5-minute write at 1.25x the input rate and a 1-hour write at 2.00x, so pricing every write at the
5-minute rate understates a 1-hour write by 37.5%.

In [ ]:
# ---- Part A: can cache spend alone breach a budget? ----
CACHE_ENF_USER = "example-cache-heavy"
ce_arn = f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/{DEMO_ROLE}/{CACHE_ENF_USER}"
CE_LIMIT = Decimal("0.25")

budget_tbl.put_item(Item={"user_id": CACHE_ENF_USER, "daily_limit_usd": CE_LIMIT,
                          "monthly_limit_usd": Decimal("1000"),
                          "alert_thresholds": [Decimal("0.5")]})
usage_tbl.put_item(Item={"user_id": CACHE_ENF_USER, "daily_spend_usd": Decimal("0"),
                         "monthly_spend_usd": Decimal("0"), "daily_invocation_count": 0,
                         "is_denied": False, "iam_role_name": DEMO_ROLE,
                         "iam_user_name": "", "principal_type": "role"})
try:
    iam.delete_role_policy(RoleName=DEMO_ROLE, PolicyName=f"BudgetExceeded-{CACHE_ENF_USER}")
except ClientError:
    pass

# Trivial new input, trivial output, 100K tokens written to cache.
rec = make_record(9800, tokens_in=50, tokens_out=50, cache_write=100_000, arn=ce_arn)
naive = (Decimal(50) / 1000) * p_in + (Decimal(50) / 1000) * p_out

send([rec])
if DELIVERY == "logs":
    time.sleep(20)

ce = usage_tbl.get_item(Key={"user_id": CACHE_ENF_USER}).get("Item", {})
ce_spend = Decimal(str(ce.get("daily_spend_usd", 0)))

print("Cache-heavy request: 50 in / 50 out / 100,000 cache write")
print(f"  daily limit                 ${CE_LIMIT}")
print(f"  envelope-only would charge  ${naive:.6f}   ({naive / CE_LIMIT * 100:.2f}% of limit)")
print(f"  actually charged            ${ce_spend:.6f}   ({ce_spend / CE_LIMIT * 100:.1f}% of limit)")
print(f"  cache write tokens metered  {int(ce.get('daily_cache_write_tokens', 0)):,}")
print(f"  is_denied                   {ce.get('is_denied')}")

enforced = bool(ce.get("is_denied"))
print()
if enforced and naive < CE_LIMIT:
    print("PASS: a single cache-heavy request breached the budget and was enforced.")
    print("      Under envelope-only metering it would have registered as "
          f"{naive / CE_LIMIT * 100:.2f}% of the limit and never triggered.")
elif not enforced:
    print("FAIL: cache spend did not trigger enforcement. Check that the deployed")
    print("      Lambda parses the usage object and that cache pricing is populated.")

# Release the demo user so the role does not accumulate policies
try:
    iam.delete_role_policy(RoleName=DEMO_ROLE, PolicyName=f"BudgetExceeded-{CACHE_ENF_USER}")
    print("\n(cleaned up the deny policy for this test user)")
except ClientError:
    pass


# ---- Part B: is every response shape classified correctly? ----
SHAPE_USER = "example-shape-probe"
sp_arn = f"arn:aws:sts::{ACCOUNT_ID}:assumed-role/{DEMO_ROLE}/{SHAPE_USER}"

budget_tbl.put_item(Item={"user_id": SHAPE_USER, "daily_limit_usd": Decimal("1000"),
                          "monthly_limit_usd": Decimal("10000"),
                          "alert_thresholds": [Decimal("0.99")]})


def make_openai_record(seq, total_in, cached, written, out):
    """OpenAI-compatible shape: input_tokens INCLUDES cache tokens."""
    r = make_record(seq, tokens_in=total_in, tokens_out=out, arn=sp_arn,
                    operation="InvokeModel", include_usage=False)
    r["output"]["outputBodyJson"] = {"usage": {
        "input_tokens": total_in,
        "output_tokens": out,
        "input_tokens_details": {"cached_tokens": cached, "cache_write_tokens": written},
    }}
    return r


def make_anthropic_record(seq, residual_in, cache_read, cache_write, out):
    r = make_record(seq, tokens_in=residual_in, tokens_out=out, arn=sp_arn,
                    operation="InvokeModel", include_usage=False)
    r["output"]["outputBodyJson"] = {"usage": {
        "input_tokens": residual_in, "output_tokens": out,
        "cache_read_input_tokens": cache_read,
        "cache_creation_input_tokens": cache_write,
    }}
    return r


# label, record, expected source, expected cache_read metered
SHAPES = [
    ("Converse (usage object)",
     make_record(9810, tokens_in=500, tokens_out=200, cache_read=4000, arn=sp_arn),
     "converse", 4000),
    ("Anthropic native",
     make_anthropic_record(9811, 350, 12000, 0, 200),
     "anthropic_native", 12000),
    ("OpenAI (input_tokens INCLUDES cache)",
     make_openai_record(9812, 3671, 3626, 0, 100),
     "openai", 3626),
    ("Envelope only (no usage object)",
     make_record(9813, tokens_in=2000, tokens_out=900, arn=sp_arn, include_usage=False),
     "envelope_only", 0),
]

print(f"\n\n{'shape':<38} {'source recorded':<18} {'cache read':>11}  ok")
print("-" * 78)

shapes_ok = True
for label, record, want_source, want_cache in SHAPES:
    usage_tbl.put_item(Item={"user_id": SHAPE_USER, "daily_spend_usd": Decimal("0"),
                             "monthly_spend_usd": Decimal("0"), "daily_invocation_count": 0,
                             "is_denied": False, "iam_role_name": DEMO_ROLE,
                             "iam_user_name": "", "principal_type": "role"})
    send([record])
    if DELIVERY == "logs":
        time.sleep(20)

    it = usage_tbl.get_item(Key={"user_id": SHAPE_USER}).get("Item", {})
    got_source = it.get("last_usage_source", "(not set)")
    got_cache = int(it.get("daily_cache_read_tokens", 0))
    ok = got_source == want_source and got_cache == want_cache
    shapes_ok = shapes_ok and ok
    print(f"{label:<38} {got_source:<18} {got_cache:>11,}  {'PASS' if ok else 'FAIL'}")

print("-" * 78)
if shapes_ok:
    print("PASS: all four response shapes classified correctly.")
    print("      The OpenAI row is the important one — its input_tokens includes cache")
    print("      tokens, so the non-cached remainder must be derived by subtraction")
    print("      (3671 - 3626 = 45) rather than added, or the prefix is counted twice.")
else:
    print("FAIL: at least one shape was misclassified. Redeploy and rerun.")


# ---- Part C: are 5-minute and 1-hour cache writes priced differently? ----
# Anthropic bills a 5m cache write at 1.25x input and a 1h write at 2.00x, so
# pricing every write at the 5m rate understates a 1h write by 37.5%.

# Find a model that actually has a published 1-hour rate. Only Claude Opus 4.5,
# Sonnet 4.5 and Haiku 4.5 support that tier.
ttl_model, ttl_prices = None, None
scan_kw = {"FilterExpression": "attribute_exists(cache_write_1h_price_per_1k_tokens)"}
resp = pricing_tbl.scan(**scan_kw)
for item in sorted(resp.get("Items", []), key=lambda i: i["model_id"]):
    if item["model_id"].startswith("us."):     # prefer a cross-region profile ID
        ttl_model, ttl_prices = item["model_id"], item
        break
if ttl_model is None and resp.get("Items"):
    it0 = sorted(resp["Items"], key=lambda i: i["model_id"])[0]
    ttl_model, ttl_prices = it0["model_id"], it0

print("\n\n--- 5-minute vs 1-hour cache writes ---")
if ttl_model is None:
    print("No model in the pricing table has a 1-hour cache rate yet.")
    print("Run scripts/seed_pricing.py then the PricingSync Lambda, and rerun.")
else:
    t_in = Decimal(str(ttl_prices["input_price_per_1k_tokens"]))
    t_out = Decimal(str(ttl_prices["output_price_per_1k_tokens"]))
    t_w5m = Decimal(str(ttl_prices["cache_write_price_per_1k_tokens"]))
    t_w1h = Decimal(str(ttl_prices["cache_write_1h_price_per_1k_tokens"]))
    print(f"{ttl_model}")
    print(f"  source={ttl_prices.get('pricing_service_code')} "
          f"scope={ttl_prices.get('rate_scope')} "
          f"cache_src={ttl_prices.get('cache_pricing_source')}")
    print(f"  input={t_in}  write5m={t_w5m} ({t_w5m / t_in}x)  "
          f"write1h={t_w1h} ({t_w1h / t_in}x)\n")

    def make_ttl_record(seq, w5m, w1h, tin=100, tout=100):
        """Converse record carrying an explicit per-TTL cacheDetails breakdown."""
        rec = make_record(seq, model_id=ttl_model, tokens_in=tin, tokens_out=tout,
                          arn=sp_arn, cache_write=w5m + w1h)
        details = []
        if w1h:
            details.append({"inputTokens": w1h, "ttl": "1h"})
        if w5m:
            details.append({"inputTokens": w5m, "ttl": "5m"})
        rec["output"]["outputBodyJson"]["usage"]["cacheDetails"] = details
        return rec

    TOKENS = 20_000
    base = (Decimal(100) / 1000) * t_in + (Decimal(100) / 1000) * t_out
    TTL_CASES = [
        ("all 5-minute TTL", TOKENS, 0, base + (Decimal(TOKENS) / 1000) * t_w5m),
        ("all 1-hour TTL", 0, TOKENS, base + (Decimal(TOKENS) / 1000) * t_w1h),
        ("split 10k / 10k", TOKENS // 2, TOKENS // 2,
         base + (Decimal(TOKENS // 2) / 1000) * t_w5m
         + (Decimal(TOKENS // 2) / 1000) * t_w1h),
    ]

    print(f"{'scenario':<22} {'charged':>12} {'expected':>12} {'1h tokens':>11}  ok")
    print("-" * 66)
    ttl_ok = True
    charged_by_case = {}
    for n, (label, w5m, w1h, expected) in enumerate(TTL_CASES, start=9820):
        usage_tbl.put_item(Item={"user_id": SHAPE_USER, "daily_spend_usd": Decimal("0"),
                                 "monthly_spend_usd": Decimal("0"),
                                 "daily_invocation_count": 0, "is_denied": False,
                                 "iam_role_name": DEMO_ROLE, "iam_user_name": "",
                                 "principal_type": "role"})
        send([make_ttl_record(n, w5m, w1h)])
        if DELIVERY == "logs":
            time.sleep(20)
        it = usage_tbl.get_item(Key={"user_id": SHAPE_USER}).get("Item", {})
        charged = Decimal(str(it.get("daily_spend_usd", 0)))
        tracked_1h = int(it.get("daily_cache_write_1h_tokens", 0))
        ok = charged == expected and tracked_1h == w1h
        ttl_ok = ttl_ok and ok
        charged_by_case[label] = charged
        print(f"{label:<22} ${charged:>11.6f} ${expected:>11.6f} {tracked_1h:>11,}  "
              f"{'PASS' if ok else 'FAIL'}")

    print("-" * 66)
    c5m = charged_by_case["all 5-minute TTL"]
    c1h = charged_by_case["all 1-hour TTL"]
    if c1h > 0:
        print(f"Same {TOKENS:,} write tokens cost ${c5m} at 5m and ${c1h} at 1h.")
        print(f"Pricing the 1h request at the 5m rate would understate it by "
              f"{(c1h - c5m) / c1h * 100:.1f}%.")
    print()
    print("PASS: TTL tiers priced separately." if ttl_ok
          else "FAIL: TTL pricing mismatch. Check cacheDetails parsing and redeploy.")

---
## 12. Cleanup

Removes everything the demo created. Leaves the deployed stack and the seeded pricing table alone.

In [ ]:
CONFIRM = False   # flip to True to actually clean up

if not CONFIRM:
    print("Set CONFIRM = True to run cleanup.")
else:
    for pn in list(iam.list_role_policies(RoleName=DEMO_ROLE)["PolicyNames"]):
        iam.delete_role_policy(RoleName=DEMO_ROLE, PolicyName=pn)
        print(f"Deleted inline policy {pn}")
    try:
        iam.delete_role(RoleName=DEMO_ROLE)
        print(f"Deleted role {DEMO_ROLE}")
    except ClientError as e:
        print(f"Could not delete role: {e}")

    for uid in (DEMO_USER_ID, globals().get("PROBE_USER"), globals().get("CC_USER"),
                globals().get("CACHE_ENF_USER"), globals().get("SHAPE_USER")):
        if not uid:
            continue
        usage_tbl.delete_item(Key={"user_id": uid})
        budget_tbl.delete_item(Key={"user_id": uid})
        print(f"Removed usage + budget rows for {uid}")

    print("\nDedup entries left in place — they expire on their own via TTL (24h).")
    print("Cleanup complete.")

---
## Appendix — design notes and limitations

### What this control is

A **fast detective control with automated response**, not a preventive one. Spend is measured *after*
the invocation completes, from the invocation log. End-to-end reaction is roughly 30–60s: 5–30s for
Bedrock to deliver the log, plus subscription filter and Lambda time. A user can overshoot their limit
within that window, and a single large-context streaming request can overshoot substantially. A hard
preventive cap requires an inline proxy or gateway in the request path.

### Hard dependency: model invocation logging

Documented scope is the `bedrock-runtime` endpoint and four operations: `Converse`, `ConverseStream`,
`InvokeModel`, `InvokeModelWithResponseStream`. The `bedrock-mantle` endpoint — OpenAI Chat Completions,
OpenAI Responses, and Anthropic Messages — is explicitly **not** captured by invocation logging, so that
traffic is invisible to this solution. Mantle has CloudWatch metrics and CloudTrail (as a paid *data*
event), but no per-request token payload in a log stream to meter against.

### Coverage gaps

| Gap | Effect |
|---|---|
| `bedrock-mantle` traffic | Not logged, not metered, not enforced |
| Image / video models | No token counts, prices at $0.00 |
| Imported (CMI) models | Billed per model-copy-minute, not per token |
| Provisioned Throughput | Billed per model-unit-hour, not per token |
| Prompt caching | **Fixed.** Cache reads and writes are parsed from the response body and priced from authoritative Price List rates, with 5-minute and 1-hour TTL tiers priced separately — see section 11 |
| Batch inference | Discounted ~50%; pricing sync deliberately skips batch usage types |
| Guardrails / KB / Agents | Separately billed, not captured |

### Shared-role blast radius — fixed

`user_id` comes from the **session name** in `identity.arn`, but IAM inline policies attach to the
**role**. The original deny used `Resource: "*"` with no principal condition, so on a shared role — the
normal IAM Identity Center pattern — one user breaching their budget would have denied Bedrock to
**everyone on that role**.

The deny is now scoped to the specific session:

```json
{
  "Effect": "Deny",
  "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream",
             "bedrock:Converse", "bedrock:ConverseStream"],
  "Resource": "*",
  "Condition": {"StringLike": {"aws:userid": "AROAEXAMPLEROLEID:demo-user-01"}}
}
```

`aws:userid` for an assumed role is `<role-unique-id>:<session-name>`, read via `iam:GetRole`. If that
lookup fails and `REQUIRE_SCOPED_DENY` is true (the default), the Lambda **declines to enforce** and
raises the `UnscopableDenySkipped` alarm rather than attaching an unscoped deny. Failing to block one
user beats blocking a whole team.

Still relevant at scale: IAM caps aggregate inline policy size at 10,240 characters per role and 2,048
per user, so a shared role accommodates roughly 30 concurrently denied users before `PutRolePolicy`
starts failing. Beyond that, consolidate into one policy listing multiple session names in a single
condition.

### Scale considerations

- **Lambda concurrency** is capped at 50 reserved. Subscription-filter delivery is asynchronous;
  sustained throttling drops events after async retries, and dropped events mean unmetered spend and
  silent enforcement failure. *Mitigated:* an SQS DLQ captures failures, and the `EnforcementThrottled`,
  `EnforcementErrors` and `EnforcementDLQNotEmpty` alarms make the condition visible. Raise reserved
  concurrency before sustained throttling.
- **Per-record DynamoDB work** is 1 conditional dedup write, 1 usage update, 1 pricing read and 1
  budget-config read. *Mitigated:* budget config is now cached in-process for 60s, so it is no longer the
  hottest read. Budget changes take effect within a minute.
- **Budget reset** uses a filtered `Scan`, which reads every item before filtering. *Mitigated:* it now
  projects only the needed attributes and has a 5-minute timeout, comfortable into the low tens of
  thousands of users. Beyond that, move to a sparse GSI or a segmented parallel scan.
- **Threshold alerts** *now* use an atomic conditional `ADD` to an `alerts_sent` string set, so exactly
  one alert fires per threshold per period regardless of concurrency. The set is cleared on reset.
- **Enforcement** *now* claims the deny with a conditional update, so concurrent invocations cannot
  double-enforce, and the claim is rolled back if the IAM attach fails.
- **Hot partition**: all of a user's writes target one item. DynamoDB allows ~1,000 WCU/s per partition
  key, which a busy agentic workload could approach.
- **IAM throttling**: `PutRolePolicy` has modest rate limits, and IAM is eventually consistent. A
  correlated mass-enforcement event (a pricing regression, say) becomes a throttling storm.
- **Log group quota**: 2 subscription filters per log group. If these logs are already streamed to a
  SIEM or similar consumer, only one slot may remain.

### Prompt caching, in one line

The control prices five components: non-cached input, output, cache reads, 5-minute cache writes and
1-hour cache writes. Metering only the log envelope — as the original implementation did — understates
agentic coding assistants by multiples, because `inputTokenCount` excludes cache tokens and cached
context dominates their volume. Rates come from whichever Price List service code publishes the model,
falling back to per-family multipliers that are flagged via `EstimatedCachePricing` rather than applied
silently. Section 11 verifies this against the deployed stack and quantifies what was being missed.

### Roadmap suggestion: `requestMetadata`

Bedrock now supports caller-supplied `requestMetadata` key-value tags on `Converse`/`ConverseStream` and
(since May 2026) `InvokeModel`/`InvokeModelWithResponseStream`. Those tags land in the invocation log.
Attributing on `requestMetadata` (team, project, environment) is far more robust than inferring identity
from an STS session name, and it survives shared roles and service-to-service call patterns. Worth
positioning as the v2 attribution key, with `identity.arn` as the fallback.